### T-ALL - Multivariate Cox Regression Pipeline

Purpose : 
Multivariate Cox tests whether each variable retains independent prognostic value AFTER adjusting for all other variables simultaneously. A variable significant in univariate Cox may lose significane here because it was confounded by another variable (e.g. ETP status and MRD are correlated - only one may carry independent signal).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import (logrank_test, proportional_hazard_test,
                                   multivariate_logrank_test)
from lifelines.utils import concordance_index
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings, os, re
from itertools import combinations
 
warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 150, 'font.size': 10,
    'axes.titlesize': 11, 'figure.facecolor': 'white'
})

In [2]:
INPUT_CSV   = "09_TARGET_model_ready.csv"
COX_DIR     = "TARGET_univariate_cox_outputs"
UMAP_DIR    = "TARGET_umap_outputs"
OUTPUT_DIR  = "TARGET_multivariate_cox_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
# Survival Endpoints
ALL_ENDPOINTS = [
    ("OS",  "OS.status",  "Overall Survival (OS)"),
    ("EFS", "EFS.status", "Event-Free Survival (EFS)"),
    ("DFS", "DFS.status", "Disease-Free Survival (DFS)"),
]
PRIMARY_DURATION = "OS"
PRIMARY_EVENT    = "OS.status"
 
# Model selection settings
VIF_THRESHOLD       = 5.0    
P_ENTRY             = 0.05  
P_REMOVE            = 0.10  
MIN_EVENTS_PER_VAR  = 10 

#### STEP 0: LOAD DATA, RECOVER COLUMN NAMES, MERGE UMAP CLUSTERS

Need to keep the same name-recovery logic as the UMAP script so the three scripts (Cox → UMAP → Multivariate) remain a coherent pipeline. The Cox display names (with spaces) are mapped back to df column names (dots).  

In [3]:
df = pd.read_csv(INPUT_CSV)
print(f"  Data shape: {df.shape[0]} patients × {df.shape[1]} columns")
 
# Coerce all survival columns to numeric
for dur, evt, _ in ALL_ENDPOINTS:
    for col in [dur, evt]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
 
# ── Variable definitions (from T-ALL_COX.ipynb) ──
CONTINUOUS_VARS = [
    "Age.at.Diagnosis.in.Years",
    "WBC_log",
    "Percent.Blasts.Tumor.Sample.Diagnostic",
    "Germline.with.Tumor.Contamination.%.Blasts",
    "Day.29.MRD",                          # continuous in Cox notebook
]
 
BINARY_ORDINAL_VARS = [
    "Sex",
    "ETP.STATUS",
    "CNS.Status",
    "End.of.Consolidation.MRD",
    "Day.29.morphologic.Response",
    "Had_Relapse_Specimen",
    "In.TARGET.Cohort.(n=.264).RNASeq.and.WES",
    # Day29_MRD_tested and Death_recorded intentionally excluded
]
 
known_non_feature = (
    CONTINUOUS_VARS + BINARY_ORDINAL_VARS +
    [d for d, e, _ in ALL_ENDPOINTS] +
    [e for d, e, _ in ALL_ENDPOINTS] +
    ["USI", "patient_id", "Day29_MRD_tested", "Death_recorded"]
)
 
DUMMY_COLS = [
    c for c in df.columns
    if c not in known_non_feature
    and "_" in c
    and df[c].dtype in ['int64', 'float64', 'int32', 'uint8']
    and df[c].nunique() <= 3
]
 
ALL_CANDIDATES = [c for c in CONTINUOUS_VARS + BINARY_ORDINAL_VARS + DUMMY_COLS
                  if c in df.columns]
 
# ── Name-recovery helper (reverse Cox display-name cleaning) ──
def display_to_col(display_name, candidate_cols):
    """Reverse var.replace('.', ' ').replace('_', ' ') to get df column name."""
    lookup = {c.replace(".", " ").replace("_", " ").strip().lower(): c
              for c in candidate_cols}
    return lookup.get(display_name.strip().lower())
 
# ── Load significant variable lists from Cox outputs ──
def load_sig_cols(ep_safe_label):
    path = f"{COX_DIR}/05_significant_vars_{ep_safe_label}.csv"
    if not os.path.exists(path):
        print(f"  ⚠ Not found: {path}")
        return []
    display_names = pd.read_csv(path)['variable'].tolist()
    recovered = []
    for dn in display_names:
        col = display_to_col(dn, ALL_CANDIDATES)
        if col:
            recovered.append(col)
        else:
            print(f"  ⚠ Could not recover column for: '{dn}'")
    return recovered
 
sig_os  = load_sig_cols("Overall_Survival_OS")
sig_efs = load_sig_cols("Event-Free_Survival_EFS")
sig_dfs = load_sig_cols("Disease-Free_Survival_DFS")
 
print(f"\n  Significant vars — OS:  {sig_os}")
print(f"  Significant vars — EFS: {sig_efs}")
print(f"  Significant vars — DFS: {sig_dfs}")
 
# ── Diagnostic: warn if any endpoint has only 1 or 0 vars ──
for ep_name, ep_vars in [("OS", sig_os), ("EFS", sig_efs), ("DFS", sig_dfs)]:
    if len(ep_vars) == 0:
        print(f"  ⚠ WARNING: No significant vars recovered for {ep_name}. "
              f"Check that {COX_DIR}/05_significant_vars_*.csv exists and column names match.")
    elif len(ep_vars) == 1:
        print(f"  ⚠ WARNING: Only 1 significant var for {ep_name} — "
              f"multivariate model will be trivially univariate. "
              f"Check name-recovery logic or lower P_ENTRY threshold.")
 
# ── Load PH violation list ──
ph_path = f"{COX_DIR}/03_ph_assumption_check.csv"
ph_violated = []
if os.path.exists(ph_path):
    ph_df_raw = pd.read_csv(ph_path)
    violated_display = ph_df_raw[~ph_df_raw['ph_assumption_ok']]['variable'].tolist()
    ph_violated = [display_to_col(v, ALL_CANDIDATES) for v in violated_display
                   if display_to_col(v, ALL_CANDIDATES)]
    if ph_violated:
        print(f"\n  ⚠ PH-violated variables (will be stratified): {ph_violated}")
 
# ── Merge UMAP cluster labels ──
cluster_path = f"{UMAP_DIR}/07_cluster_labels.csv"
umap_cluster_col = None
if os.path.exists(cluster_path):
    cluster_df = pd.read_csv(cluster_path)
    id_col = "USI" if "USI" in cluster_df.columns else cluster_df.columns[0]
    df = df.merge(cluster_df[[id_col, "KMeans_cluster"]],
                  on=id_col, how='left')
    df["KMeans_cluster"] = pd.to_numeric(df["KMeans_cluster"], errors='coerce')
    umap_cluster_col = "KMeans_cluster"
    print(f"\n  UMAP KMeans clusters merged. Distribution:")
    print(f"  {df['KMeans_cluster'].value_counts().sort_index().to_dict()}")
else:
    print(f"\n  ⚠ UMAP cluster file not found at {cluster_path} — skipping")
 
# ── Working dataframe ──
df_model = df.dropna(subset=[PRIMARY_DURATION, PRIMARY_EVENT]).copy()
print(f"\n  Working dataset: {len(df_model)} patients, "
      f"{int(df_model[PRIMARY_EVENT].sum())} OS events "
      f"({df_model[PRIMARY_EVENT].mean()*100:.1f}% event rate)")
 
# ── Events-per-variable rule ──
n_events = int(df_model[PRIMARY_EVENT].sum())
max_vars  = n_events // MIN_EVENTS_PER_VAR
print(f"\n  Max variables in OS model (rule of ≥{MIN_EVENTS_PER_VAR} events/var): "
      f"{max_vars}")
if len(sig_os) > max_vars:
    print(f"  ⚠ {len(sig_os)} significant OS vars exceeds limit of {max_vars}.")
    print(f"  → Model selection (Step 3) will reduce the model.")

  Data shape: 1335 patients × 30 columns

  Significant vars — OS:  ['Day.29.MRD']
  Significant vars — EFS: ['Day.29.MRD']
  Significant vars — DFS: ['Day.29.MRD']
  ⚠ WARNING: Only 1 significant var for OS — multivariate model will be trivially univariate. Check name-recovery logic or lower P_ENTRY threshold.
  ⚠ WARNING: Only 1 significant var for EFS — multivariate model will be trivially univariate. Check name-recovery logic or lower P_ENTRY threshold.
  ⚠ WARNING: Only 1 significant var for DFS — multivariate model will be trivially univariate. Check name-recovery logic or lower P_ENTRY threshold.

  UMAP KMeans clusters merged. Distribution:
  {0: 710, 1: 247, 2: 378}

  Working dataset: 1335 patients, 141 OS events (10.6% event rate)

  Max variables in OS model (rule of ≥10 events/var): 14


#### STEP 1: MULTICOLLINEARITY CHECK (VIF)

Cox regression, like all regression models, assumes predictors are not strongly collinear. When two variables are highly correlated (e.g. Day.29.MRD and End.of.Consolidation.MRD), including both inflates standard errors, makes HRs unstable, and causes convergence problems.

VIF (Variance Inflation Factor): 
- VIF = 1   → no correlation with other predictors
- VIF 1-5   → acceptable
- VIF 5-10  → concerning - consider dropping one variable
- VIF > 10  → severe multicollinearity - must act

In [4]:
def compute_vif(df_in, feature_cols):
    """Compute VIF for each feature. Requires complete-case numeric matrix."""
    subset = df_in[feature_cols].dropna()
    # Need all numeric
    for col in feature_cols:
        subset[col] = pd.to_numeric(subset[col], errors='coerce')
    subset = subset.dropna()
 
    if len(subset) < len(feature_cols) + 5:
        print(f"  ⚠ Too few complete cases ({len(subset)}) for VIF — skipping")
        return pd.DataFrame()
 
    # Add constant for VIF calculation
    X = subset.values
    vif_data = []
    for i, col in enumerate(feature_cols):
        try:
            vif = variance_inflation_factor(X, i)
            vif_data.append({"variable": col,
                              "display": col.replace(".", " ").replace("_", " "),
                              "VIF": round(vif, 2)})
        except Exception:
            vif_data.append({"variable": col,
                              "display": col.replace(".", " ").replace("_", " "),
                              "VIF": np.nan})
    return pd.DataFrame(vif_data).sort_values("VIF", ascending=False)
 
# Run VIF on all significant OS variables (main model)
vif_cols = [c for c in sig_os if c in df_model.columns]
if len(vif_cols) >= 2:
    # Impute for VIF (VIF needs complete matrix)
    imputer_vif = SimpleImputer(strategy='median')
    X_for_vif   = pd.DataFrame(
        imputer_vif.fit_transform(df_model[vif_cols]),
        columns=vif_cols
    )
    vif_df = compute_vif(X_for_vif, vif_cols)
 
    print("  VIF Results:")
    print(f"  {'Variable':<50} {'VIF':>6}")
    print("  " + "-" * 58)
    for _, row in vif_df.iterrows():
        flag = " ⚠ HIGH" if row['VIF'] > VIF_THRESHOLD else ""
        print(f"  {row['display']:<50} {row['VIF']:>6.2f}{flag}")
 
    vif_df.to_csv(f"{OUTPUT_DIR}/01_vif_check.csv", index=False)
 
    # Flag high-VIF variables for review
    high_vif = vif_df[vif_df['VIF'] > VIF_THRESHOLD]['variable'].tolist()
    if high_vif:
        print(f"\n  ⚠ High VIF variables: {high_vif}")
        print("  Strategy: Keep the variable with stronger univariate HR,")
        print("  or the one with clearer biological interpretation.")
        print("  (Do not remove automatically — this requires clinical judgment)")
    else:
        print("\n  ✓ All VIF values acceptable (< 5)")
 
    # ── VIF bar chart ──
    fig, ax = plt.subplots(figsize=(8, max(4, len(vif_df)*0.4 + 1.5)))
    colors = ['#C0392B' if v > VIF_THRESHOLD else '#2980B9'
              for v in vif_df['VIF']]
    ax.barh(vif_df['display'], vif_df['VIF'], color=colors, alpha=0.85,
            edgecolor='white')
    ax.axvline(VIF_THRESHOLD, color='red', ls='--', lw=1.5,
               label=f'VIF={VIF_THRESHOLD} threshold')
    ax.axvline(1, color='green', ls=':', lw=1)
    ax.set_xlabel("Variance Inflation Factor (VIF)")
    ax.set_title("Multicollinearity Check — Significant OS Variables\n"
                 "VIF > 5 = multicollinearity concern", fontsize=11)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/01_vif_chart.png", bbox_inches='tight')
    plt.close()
    print(f"\n  VIF chart → {OUTPUT_DIR}/01_vif_chart.png")
else:
    print("  Fewer than 2 significant OS variables — VIF check skipped")
    high_vif = []
 
# #### STEP 2: FIT FULL MULTIVARIATE COX MODEL PER ENDPOINT
 
def safe_col_name(col):
    """Wrap column name in backticks if it contains special characters."""
    if re.search(r'[\s\-\(\)\.\%]', col):
        return f'`{col}`'
    return col
 
def fit_cox_model(df_in, feature_cols, duration_col, event_col,
                  strata_cols=None, penalizer=0.1, label=""):
    """
    Fit a multivariate Cox model. Returns (cph_object, summary_df) or None.
 
    penalizer=0.1: small ridge penalty for numerical stability in small
    cohorts. This is NOT LASSO — it does not zero out coefficients. It
    slightly shrinks all HRs toward 1, which stabilises convergence
    without meaningfully biasing the estimates for a small T-ALL cohort.
 
    strata_cols: list of columns to stratify on (PH-violated variables).
    These are adjusted for but do not get a HR in the output.
    """
    # Build complete-case subset
    cols_needed = feature_cols + [duration_col, event_col]
    if strata_cols:
        cols_needed += strata_cols
    subset = df_in[[c for c in cols_needed if c in df_in.columns]].dropna()
 
    # Coerce to numeric
    for col in feature_cols:
        if col in subset.columns:
            subset[col] = pd.to_numeric(subset[col], errors='coerce')
    subset = subset.dropna()
 
    n_patients = len(subset)
    n_events   = int(subset[event_col].sum())
    print(f"  {label}: n={n_patients}, events={n_events}")
 
    if n_patients < 20 or n_events < len(feature_cols) * 5:
        print(f"  ⚠ Insufficient data for {len(feature_cols)}-variable model")
        return None, None
 
    # Build formula (backtick-quote special chars)
    formula_vars = [c for c in feature_cols
                    if c not in (strata_cols or [])]
    formula = " + ".join(safe_col_name(c) for c in formula_vars)
 
    cph = CoxPHFitter(penalizer=penalizer)
    try:
        cph.fit(subset,
                duration_col=duration_col,
                event_col=event_col,
                formula=formula,
                strata=strata_cols)
 
        # Post-fit check: coefficient explosion = separation / convergence fail
        if (cph.summary['coef'].abs() > 10).any():
            print(f"  ⚠ Coefficient explosion detected — increasing penalizer")
            cph = CoxPHFitter(penalizer=1.0)
            cph.fit(subset, duration_col=duration_col,
                    event_col=event_col, formula=formula,
                    strata=strata_cols)
 
        c_idx   = cph.concordance_index_
        log_lik = cph.log_likelihood_
 
        summary = cph.summary[['coef', 'exp(coef)',
                                'exp(coef) lower 95%',
                                'exp(coef) upper 95%',
                                'z', 'p']].copy()
        summary.columns = ['coef', 'HR', 'HR_lower95', 'HR_upper95', 'z', 'p_value']
        summary = summary.round(4)
        summary['significant'] = summary['p_value'] < 0.05
        summary['display']     = summary.index.str.replace(".", " ").str.replace("_", " ")
        summary['n']           = n_patients
        summary['n_events']    = n_events
        summary['C_index']     = round(c_idx, 3)
 
        print(f"  C-index = {c_idx:.3f}  |  log-likelihood = {log_lik:.2f}")
        return cph, summary
 
    except Exception as ex:
        print(f"  ✗ Model failed: {ex}")
        return None, None
 
# Storage for all fitted models
full_models   = {}   # endpoint label → (cph, summary)
full_summaries = {}
 
for duration_col, event_col, ep_label in ALL_ENDPOINTS:
    print(f"\n  ── {ep_label} ──")
 
    # Select significant variables for this endpoint
    if ep_label == "Overall Survival (OS)":
        ep_sig_cols = sig_os
    elif ep_label == "Event-Free Survival (EFS)":
        ep_sig_cols = sig_efs
    else:
        ep_sig_cols = sig_dfs
 
    ep_sig_cols = [c for c in ep_sig_cols if c in df_model.columns]
 
    # Add UMAP cluster if available and not already included
    if (umap_cluster_col and umap_cluster_col in df_model.columns
            and umap_cluster_col not in ep_sig_cols):
        ep_sig_cols.append(umap_cluster_col)
        print(f"  + Added UMAP KMeans cluster to {ep_label} model")
 
    if not ep_sig_cols:
        print(f"  No significant variables — skipping {ep_label}")
        continue
 
    # Determine strata (PH-violated vars that are in this model)
    strata = [c for c in ph_violated if c in ep_sig_cols] or None
    model_vars = [c for c in ep_sig_cols
                  if c not in (strata or [])]
 
    if strata:
        print(f"  Stratifying on (PH-violated): {strata}")
 
    cph, summary = fit_cox_model(
        df_model, ep_sig_cols, duration_col, event_col,
        strata_cols=strata, penalizer=0.1, label=ep_label
    )
 
    if summary is not None:
        full_models[ep_label]    = cph
        full_summaries[ep_label] = summary
 
        # Print results
        print(f"\n  {'Variable':<45} {'HR':>7} {'95% CI':>18} {'p':>8}")
        print("  " + "-" * 82)
        for idx, row in summary.sort_values('p_value').iterrows():
            sig = " *" if row['p_value'] < 0.05 else "  "
            print(f"  {row['display']:<45} "
                  f"{row['HR']:>7.3f} "
                  f"[{row['HR_lower95']:.3f}–{row['HR_upper95']:.3f}]{sig:>3} "
                  f"p={row['p_value']:.4f}")
 
        safe_label = ep_label.replace(" ", "_").replace("(","").replace(")","")
        summary.to_csv(f"{OUTPUT_DIR}/02_full_model_{safe_label}.csv")
        print(f"\n  Full model saved → {OUTPUT_DIR}/02_full_model_{safe_label}.csv")

  Fewer than 2 significant OS variables — VIF check skipped

  ── Overall Survival (OS) ──
  + Added UMAP KMeans cluster to Overall Survival (OS) model
  Overall Survival (OS): n=1328, events=141
  C-index = 0.643  |  log-likelihood = -978.55

  Variable                                           HR             95% CI        p
  ----------------------------------------------------------------------------------
  Day 29 MRD                                      1.025 [1.017–1.033]  * p=0.0000
  KMeans cluster                                  0.967 [0.842–1.109]    p=0.6271

  Full model saved → TARGET_multivariate_cox_outputs/02_full_model_Overall_Survival_OS.csv

  ── Event-Free Survival (EFS) ──
  + Added UMAP KMeans cluster to Event-Free Survival (EFS) model
  Event-Free Survival (EFS): n=1328, events=220
  C-index = 0.675  |  log-likelihood = -1467.82

  Variable                                           HR             95% CI        p
  ------------------------------------------------

#### STEP 2: FIT FULL MULTIVARIATE COX MODEL PER ENDPOINT

In [5]:
def safe_col_name(col):
    """Wrap column name in backticks if it contains special characters."""
    if re.search(r'[\s\-\(\)\.\%]', col):
        return f'`{col}`'
    return col
 
def fit_cox_model(df_in, feature_cols, duration_col, event_col,
                  strata_cols=None, penalizer=0.1, label=""):
    """
    Fit a multivariate Cox model. Returns (cph_object, summary_df) or None.
 
    penalizer=0.1: small ridge penalty for numerical stability in small
    cohorts. This is NOT LASSO — it does not zero out coefficients. It
    slightly shrinks all HRs toward 1, which stabilises convergence
    without meaningfully biasing the estimates for a small T-ALL cohort.
 
    strata_cols: list of columns to stratify on (PH-violated variables).
    These are adjusted for but do not get a HR in the output.
    """
    # Build complete-case subset
    cols_needed = feature_cols + [duration_col, event_col]
    if strata_cols:
        cols_needed += strata_cols
    subset = df_in[[c for c in cols_needed if c in df_in.columns]].dropna()
 
    # Coerce to numeric
    for col in feature_cols:
        if col in subset.columns:
            subset[col] = pd.to_numeric(subset[col], errors='coerce')
    subset = subset.dropna()
 
    n_patients = len(subset)
    n_events   = int(subset[event_col].sum())
    print(f"  {label}: n={n_patients}, events={n_events}")
 
    if n_patients < 20 or n_events < len(feature_cols) * 5:
        print(f"  ⚠ Insufficient data for {len(feature_cols)}-variable model")
        return None, None
 
    # Build formula (backtick-quote special chars)
    formula_vars = [c for c in feature_cols
                    if c not in (strata_cols or [])]
    formula = " + ".join(safe_col_name(c) for c in formula_vars)
 
    cph = CoxPHFitter(penalizer=penalizer)
    try:
        cph.fit(subset,
                duration_col=duration_col,
                event_col=event_col,
                formula=formula,
                strata=strata_cols)
 
        # Post-fit check: coefficient explosion = separation / convergence fail
        if (cph.summary['coef'].abs() > 10).any():
            print(f"  ⚠ Coefficient explosion detected — increasing penalizer")
            cph = CoxPHFitter(penalizer=1.0)
            cph.fit(subset, duration_col=duration_col,
                    event_col=event_col, formula=formula,
                    strata=strata_cols)
 
        c_idx   = cph.concordance_index_
        log_lik = cph.log_likelihood_
 
        summary = cph.summary[['coef', 'exp(coef)',
                                'exp(coef) lower 95%',
                                'exp(coef) upper 95%',
                                'z', 'p']].copy()
        summary.columns = ['coef', 'HR', 'HR_lower95', 'HR_upper95', 'z', 'p_value']
        summary = summary.round(4)
        summary['significant'] = summary['p_value'] < 0.05
        summary['display']     = summary.index.str.replace(".", " ").str.replace("_", " ")
        summary['n']           = n_patients
        summary['n_events']    = n_events
        summary['C_index']     = round(c_idx, 3)
 
        print(f"  C-index = {c_idx:.3f}  |  log-likelihood = {log_lik:.2f}")
        return cph, summary
 
    except Exception as ex:
        print(f"  ✗ Model failed: {ex}")
        return None, None
 
# Storage for all fitted models
full_models   = {}   # endpoint label → (cph, summary)
full_summaries = {}
 
for duration_col, event_col, ep_label in ALL_ENDPOINTS:
    print(f"\n  ── {ep_label} ──")
 
    # Select significant variables for this endpoint
    if ep_label == "Overall Survival (OS)":
        ep_sig_cols = sig_os
    elif ep_label == "Event-Free Survival (EFS)":
        ep_sig_cols = sig_efs
    else:
        ep_sig_cols = sig_dfs
 
    ep_sig_cols = [c for c in ep_sig_cols if c in df_model.columns]
 
    # Add UMAP cluster if available and not already included
    if (umap_cluster_col and umap_cluster_col in df_model.columns
            and umap_cluster_col not in ep_sig_cols):
        ep_sig_cols.append(umap_cluster_col)
        print(f"  + Added UMAP KMeans cluster to {ep_label} model")
 
    if not ep_sig_cols:
        print(f"  No significant variables — skipping {ep_label}")
        continue
 
    # Determine strata (PH-violated vars that are in this model)
    strata = [c for c in ph_violated if c in ep_sig_cols] or None
    model_vars = [c for c in ep_sig_cols
                  if c not in (strata or [])]
 
    if strata:
        print(f"  Stratifying on (PH-violated): {strata}")
 
    cph, summary = fit_cox_model(
        df_model, ep_sig_cols, duration_col, event_col,
        strata_cols=strata, penalizer=0.1, label=ep_label
    )
 
    if summary is not None:
        full_models[ep_label]    = cph
        full_summaries[ep_label] = summary
 
        # Print results
        print(f"\n  {'Variable':<45} {'HR':>7} {'95% CI':>18} {'p':>8}")
        print("  " + "-" * 82)
        for idx, row in summary.sort_values('p_value').iterrows():
            sig = " *" if row['p_value'] < 0.05 else "  "
            print(f"  {row['display']:<45} "
                  f"{row['HR']:>7.3f} "
                  f"[{row['HR_lower95']:.3f}–{row['HR_upper95']:.3f}]{sig:>3} "
                  f"p={row['p_value']:.4f}")
 
        safe_label = ep_label.replace(" ", "_").replace("(","").replace(")","")
        summary.to_csv(f"{OUTPUT_DIR}/02_full_model_{safe_label}.csv")
        print(f"\n  Full model saved → {OUTPUT_DIR}/02_full_model_{safe_label}.csv")


  ── Overall Survival (OS) ──
  + Added UMAP KMeans cluster to Overall Survival (OS) model
  Overall Survival (OS): n=1328, events=141
  C-index = 0.643  |  log-likelihood = -978.55

  Variable                                           HR             95% CI        p
  ----------------------------------------------------------------------------------
  Day 29 MRD                                      1.025 [1.017–1.033]  * p=0.0000
  KMeans cluster                                  0.967 [0.842–1.109]    p=0.6271

  Full model saved → TARGET_multivariate_cox_outputs/02_full_model_Overall_Survival_OS.csv

  ── Event-Free Survival (EFS) ──
  + Added UMAP KMeans cluster to Event-Free Survival (EFS) model
  Event-Free Survival (EFS): n=1328, events=220
  C-index = 0.675  |  log-likelihood = -1467.82

  Variable                                           HR             95% CI        p
  ----------------------------------------------------------------------------------
  Day 29 MRD             

#### STEP 3: MODEL SELECTION - BACKWARD ELIMINATION

The full model likely contains variables that are no longer significant after mutual adjustment. Backward elimination: 
1. Fits full model
2. Finds the least significant variable (highest p-value)
3. If p > P_REMOVE threshold, removes it
4. Repeats until all remaining variables have p ≤ P_REMOVE

Why backward and not forward? In survival analysis with small n, backward selection starts from the full adjusted picture and removes noise. Forward selection risks including a variable early that becomes redundant later. Backward is the more conservative choice

In [6]:
final_models    = {}   # endpoint label → (cph, summary, selected_vars)
final_summaries = {}
 
for duration_col, event_col, ep_label in ALL_ENDPOINTS:
    if ep_label not in full_models:
        continue
 
    print(f"\n  ── {ep_label} ──")
 
    if ep_label == "Overall Survival (OS)":
        ep_sig_cols = sig_os.copy()
    elif ep_label == "Event-Free Survival (EFS)":
        ep_sig_cols = sig_efs.copy()
    else:
        ep_sig_cols = sig_dfs.copy()
 
    if umap_cluster_col and umap_cluster_col not in ep_sig_cols:
        ep_sig_cols.append(umap_cluster_col)
 
    ep_sig_cols = [c for c in ep_sig_cols if c in df_model.columns]
    strata      = [c for c in ph_violated if c in ep_sig_cols] or None
    active_vars = [c for c in ep_sig_cols if c not in (strata or [])]
 
    iteration   = 0
    aic_history = []
 
    while True:
        iteration += 1
        if not active_vars:
            print(f"  No variables remaining after iteration {iteration}")
            break
 
        cph, summary = fit_cox_model(
            df_model, active_vars + (strata or []),
            duration_col, event_col,
            strata_cols=strata, penalizer=0.1,
            label=f"  Iter {iteration} ({len(active_vars)} vars)"
        )
 
        if cph is None:
            print(f"  Model failed at iteration {iteration} — stopping")
            break
 
        # AIC = -2 * log-likelihood + 2 * k  (lower = better)
        k   = len(active_vars)
        aic = -2 * cph.log_likelihood_ + 2 * k
        aic_history.append({"iteration": iteration,
                             "n_vars": k,
                             "AIC": round(aic, 2),
                             "C_index": round(cph.concordance_index_, 3),
                             "vars": ", ".join(active_vars)})
 
        # Find worst variable (highest p-value, among non-strata vars)
        worst_p           = summary['p_value'].max()
        worst_var_display = summary['p_value'].idxmax()  # lifelines index (col name or backtick-cleaned)
 
        # Robust reverse-lookup: match the lifelines summary index against active_vars
        # by comparing stripped/normalised versions of each name.
        # lifelines may strip backticks or keep dots, so we try multiple forms.
        def _match_to_active(idx_name, candidates):
            """Return the active_vars entry that best matches the lifelines index."""
            # Direct match first (most common case — lifelines uses raw col name)
            if idx_name in candidates:
                return idx_name
            # Normalised match (dots→spaces, underscores→spaces, lower)
            norm_idx = idx_name.replace(".", " ").replace("_", " ").strip().lower()
            for c in candidates:
                if c.replace(".", " ").replace("_", " ").strip().lower() == norm_idx:
                    return c
            # Backtick-stripped match (lifelines formula interface sometimes wraps names)
            stripped = idx_name.strip("`")
            if stripped in candidates:
                return stripped
            return None
 
        worst_var = _match_to_active(worst_var_display, active_vars)
 
        print(f"    Iter {iteration}: {k} vars, AIC={aic:.2f}, "
              f"C-index={cph.concordance_index_:.3f} | "
              f"worst: '{worst_var_display}' p={worst_p:.4f}")
 
        if worst_p > P_REMOVE:
            if worst_var is not None:
                print(f"    → Removing '{worst_var}' (p={worst_p:.4f} > {P_REMOVE})")
                active_vars.remove(worst_var)
            else:
                # Last-resort: try removing by position (worst p_value row)
                # This avoids a silent stop when name normalisation fails
                fallback = next(
                    (c for c in active_vars
                     if c.replace(".", " ").replace("_", " ").strip().lower()
                        in worst_var_display.replace(".", " ").replace("_", " ").strip().lower()
                     or worst_var_display.replace(".", " ").replace("_", " ").strip().lower()
                        in c.replace(".", " ").replace("_", " ").strip().lower()),
                    None
                )
                if fallback:
                    print(f"    → Removing '{fallback}' via fallback match (p={worst_p:.4f} > {P_REMOVE})")
                    active_vars.remove(fallback)
                else:
                    print(f"    ⚠ Could not match '{worst_var_display}' to any active var — stopping safely")
                    print(f"      Active vars were: {active_vars}")
                    break
        else:
            print(f"    ✓ All remaining variables p ≤ {P_REMOVE} — model finalised")
            final_models[ep_label]    = (cph, active_vars, strata)
            final_summaries[ep_label] = summary
            break
 
        if len(active_vars) == 0:
            print("  All variables removed — null model")
            break
 
    # ── If backward elim removed everything, fall back to the last valid model ──
    if ep_label not in final_models and aic_history:
        print(f"  ℹ Backward elimination removed all vars. Using best AIC step as final model.")
        # Re-fit using the vars from the step with lowest AIC
        best_row   = min(aic_history, key=lambda x: x['AIC'])
        best_vars  = [v.strip() for v in best_row['vars'].split(',') if v.strip() in df_model.columns]
        strata_fb  = [c for c in ph_violated if c in best_vars] or None
        if best_vars:
            cph_fb, summary_fb = fit_cox_model(
                df_model, best_vars, duration_col, event_col,
                strata_cols=strata_fb, penalizer=0.1, label=f"  Fallback (k={len(best_vars)})"
            )
            if summary_fb is not None:
                final_models[ep_label]    = (cph_fb, best_vars, strata_fb)
                final_summaries[ep_label] = summary_fb
 
    # Save AIC history
    if aic_history:
        aic_df = pd.DataFrame(aic_history)
        safe_label = ep_label.replace(" ", "_").replace("(","").replace(")","")
        aic_df.to_csv(f"{OUTPUT_DIR}/03_aic_history_{safe_label}.csv", index=False)
 
        # AIC trajectory plot
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(aic_df['n_vars'], aic_df['AIC'], 'o-',
                color='#2980B9', lw=2, markersize=7)
        if ep_label in final_models:
            best_n = len(final_models[ep_label][1])
            best_aic = aic_df[aic_df['n_vars'] == best_n]['AIC'].values
            if len(best_aic):
                ax.scatter([best_n], [best_aic[0]], color='#C0392B',
                           s=120, zorder=5, label=f'Final model (k={best_n})')
        ax.set_xlabel("Number of Variables in Model")
        ax.set_ylabel("AIC (lower = better)")
        ax.set_title(f"Backward Elimination — {ep_label}\nAIC at each step",
                     fontsize=11)
        ax.legend(fontsize=9)
        ax.spines[['top', 'right']].set_visible(False)
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/03_aic_trajectory_{safe_label}.png",
                    bbox_inches='tight')
        plt.close()
 
print(f"\n  AIC trajectory plots saved to {OUTPUT_DIR}/")


  ── Overall Survival (OS) ──
    Iter 1 (2 vars): n=1328, events=141
  C-index = 0.643  |  log-likelihood = -978.55
    Iter 1: 2 vars, AIC=1961.11, C-index=0.643 | worst: 'KMeans_cluster' p=0.6271
    → Removing 'KMeans_cluster' (p=0.6271 > 0.1)
    Iter 2 (1 vars): n=1328, events=141
  C-index = 0.637  |  log-likelihood = -978.67
    Iter 2: 1 vars, AIC=1959.35, C-index=0.637 | worst: 'Day.29.MRD' p=0.0000
    ✓ All remaining variables p ≤ 0.1 — model finalised

  ── Event-Free Survival (EFS) ──
    Iter 1 (2 vars): n=1328, events=220
  C-index = 0.675  |  log-likelihood = -1467.82
    Iter 1: 2 vars, AIC=2939.63, C-index=0.675 | worst: 'KMeans_cluster' p=0.5734
    → Removing 'KMeans_cluster' (p=0.5734 > 0.1)
    Iter 2 (1 vars): n=1328, events=220
  C-index = 0.664  |  log-likelihood = -1467.98
    Iter 2: 1 vars, AIC=2937.95, C-index=0.664 | worst: 'Day.29.MRD' p=0.0000
    ✓ All remaining variables p ≤ 0.1 — model finalised

  ── Disease-Free Survival (DFS) ──
    Iter 1 (2 var

#### STEP 4: FINAL MODEL FOREST PLOTS + HR TABLES

The forest plot of the final parsimonious model is the primary figure for the manuscript. Each variable's HR is now adjusted for all others - this is the clinically meaningful estimate. 

In [7]:
def forest_plot_multivariate(summary_df, ep_label, c_index,
                               n_patients, n_events, output_path,
                               strata=None):
    """
    Publication-quality forest plot for a multivariate Cox model.
    Sorted by HR (ascending) so protective factors appear at top.
    """
    df_plot = summary_df.copy().sort_values('HR')
    n       = len(df_plot)
    if n == 0:
        return
 
    fig_h = max(5, n * 0.55 + 3)
    fig, (ax_tab, ax_for) = plt.subplots(
        1, 2, figsize=(15, fig_h),
        gridspec_kw={'width_ratios': [2.2, 3]}
    )
 
    # ── Left: text table ──
    ax_tab.set_xlim(0, 1)
    ax_tab.set_ylim(-1, n + 1)
    ax_tab.axis('off')
 
    headers = ["Variable", "n", "HR (95% CI)", "p-value"]
    col_x   = [0.0, 0.52, 0.62, 0.88]
    for hx, hdr in zip(col_x, headers):
        ax_tab.text(hx, n + 0.3, hdr, fontsize=9, fontweight='bold', va='bottom')
    ax_tab.axhline(n, color='black', lw=0.9)
 
    for i, (idx, row) in enumerate(df_plot.iterrows()):
        y      = n - 1 - i
        sig    = row['p_value'] < 0.05
        color  = '#C0392B' if sig else '#444444'
        weight = 'bold' if sig else 'normal'
 
        varname = (row['display'][:32] + '…') \
                   if len(str(row['display'])) > 34 else str(row['display'])
        ax_tab.text(col_x[0], y, varname, fontsize=8, va='center',
                    color=color, fontweight=weight)
        ax_tab.text(col_x[1], y, f"{row['n_events']}",
                    fontsize=8, va='center', ha='center', color='#555')
        ci = (f"{row['HR']:.2f} "
              f"[{row['HR_lower95']:.2f}–{row['HR_upper95']:.2f}]")
        ax_tab.text(col_x[2], y, ci, fontsize=8, va='center',
                    color=color, fontweight=weight)
        p_str = (f"{row['p_value']:.4f}" +
                 (" **" if row['p_value'] < 0.01
                  else " *" if row['p_value'] < 0.05 else ""))
        ax_tab.text(col_x[3], y, p_str, fontsize=8, va='center',
                    color=color, fontweight=weight)
 
    # Model stats below table
    strata_note = f"\nStratified on: {strata}" if strata else ""
    ax_tab.text(0, -0.7,
                f"n={n_patients}  |  Events={n_events}  |  "
                f"C-index={c_index:.3f}{strata_note}",
                fontsize=7.5, va='top', color='#555', style='italic')
 
    # ── Right: forest ──
    ax_for.set_xscale('log')
    ax_for.set_ylim(-1, n + 1)
    ax_for.axvline(1.0, color='black', lw=1.0, ls='--', zorder=1,
                   alpha=0.7)
 
    # Reference shading
    ax_for.axvspan(0.001, 1.0, alpha=0.03, color='#2980B9')
    ax_for.axvspan(1.0, 1000, alpha=0.03, color='#C0392B')
 
    for i, (idx, row) in enumerate(df_plot.iterrows()):
        y     = n - 1 - i
        sig   = row['p_value'] < 0.05
        color = '#C0392B' if sig else '#7F8C8D'
 
        # CI line
        ci_lo = max(row['HR_lower95'], 0.001)
        ci_hi = min(row['HR_upper95'], 100)
        ax_for.plot([ci_lo, ci_hi], [y, y],
                    color=color, lw=2, zorder=2, solid_capstyle='round')
        # Diamond marker for significant, square for non-significant
        marker = 'D' if sig else 's'
        ax_for.scatter(row['HR'], y, color=color, s=70,
                       zorder=3, marker=marker)
 
    ax_for.set_xlabel("Adjusted Hazard Ratio (log scale)", fontsize=9)
    ax_for.set_yticks([])
    for spine in ['left', 'right', 'top']:
        ax_for.spines[spine].set_visible(False)
 
    ax_for.text(0.03, 1.02, '← Protective', transform=ax_for.transAxes,
                fontsize=8, color='#2980B9', va='bottom')
    ax_for.text(0.97, 1.02, 'Harmful →', transform=ax_for.transAxes,
                fontsize=8, color='#C0392B', va='bottom', ha='right')
 
    sig_p   = mpatches.Patch(color='#C0392B', label='p < 0.05 (◆ diamond)')
    insig_p = mpatches.Patch(color='#7F8C8D', label='p ≥ 0.05 (■ square)')
    ax_for.legend(handles=[sig_p, insig_p], loc='lower right', fontsize=8)
 
    plt.suptitle(
        f"Multivariate Cox Regression — {ep_label}\n"
        f"Adjusted Hazard Ratios (backward elimination, p-remove={P_REMOVE})",
        fontsize=12, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    plt.savefig(output_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"  Forest plot → {output_path}")
    plt.show()
 
 
for ep_label, (cph, selected_vars, strata) in final_models.items():
    summary = final_summaries[ep_label]
    safe_label = ep_label.replace(" ", "_").replace("(","").replace(")","")
 
    subset_ep = df_model[selected_vars +
                         ([s for s in (strata or [])] if strata else []) +
                         [d for d, e, _ in ALL_ENDPOINTS if ep_label in _] +
                         [e for d, e, _ in ALL_ENDPOINTS if ep_label in _]
                         ].dropna()
    dur_col = next(d for d, e, l in ALL_ENDPOINTS if l == ep_label)
    evt_col = next(e for d, e, l in ALL_ENDPOINTS if l == ep_label)
 
    forest_plot_multivariate(
        summary_df  = summary,
        ep_label    = ep_label,
        c_index     = cph.concordance_index_,
        n_patients  = len(df_model),
        n_events    = int(df_model[evt_col].sum()),
        output_path = f"{OUTPUT_DIR}/04_forest_final_{safe_label}.png",
        strata      = strata
    )
 
    # Save final HR table
    summary.to_csv(f"{OUTPUT_DIR}/04_final_model_{safe_label}.csv")
    print(f"  Final model table → {OUTPUT_DIR}/04_final_model_{safe_label}.csv")
 
    print(f"\n  {ep_label} final variables: {selected_vars}")
 

  Forest plot → TARGET_multivariate_cox_outputs/04_forest_final_Overall_Survival_OS.png
  Final model table → TARGET_multivariate_cox_outputs/04_final_model_Overall_Survival_OS.csv

  Overall Survival (OS) final variables: ['Day.29.MRD']
  Forest plot → TARGET_multivariate_cox_outputs/04_forest_final_Event-Free_Survival_EFS.png
  Final model table → TARGET_multivariate_cox_outputs/04_final_model_Event-Free_Survival_EFS.csv

  Event-Free Survival (EFS) final variables: ['Day.29.MRD']
  Forest plot → TARGET_multivariate_cox_outputs/04_forest_final_Disease-Free_Survival_DFS.png
  Final model table → TARGET_multivariate_cox_outputs/04_final_model_Disease-Free_Survival_DFS.csv

  Disease-Free Survival (DFS) final variables: ['Day.29.MRD']


#### STEP 5: PROPORTIONAL HAZARDS CHECK ON FINAL MODEL

The full model PH assumption must be rechecked on the final model. Removing variables can change residual patterns. We use Shoenfeld residuals with rank time-transform (most robust for tied survival times, which are common in paediatric cohort data).

In [8]:
for ep_label, (cph, selected_vars, strata) in final_models.items():
    dur_col = next(d for d, e, l in ALL_ENDPOINTS if l == ep_label)
    evt_col = next(e for d, e, l in ALL_ENDPOINTS if l == ep_label)
 
    model_vars = selected_vars + (strata or [])
    subset = df_model[[c for c in model_vars + [dur_col, evt_col]
                        if c in df_model.columns]].dropna()
    for col in selected_vars:
        if col in subset.columns:
            subset[col] = pd.to_numeric(subset[col], errors='coerce')
    subset = subset.dropna()
 
    try:
        ph_result = proportional_hazard_test(cph, subset,
                                              time_transform='rank')
        print(f"  {ep_label}:")
        for var_idx, row in ph_result.summary.iterrows():
            p_ph   = float(row['p'])
            status = "✓ PH holds" if p_ph > 0.05 else "⚠ VIOLATED"
            print(f"    {str(var_idx):<45} {status}  p={p_ph:.4f}")
 
        safe_label = ep_label.replace(" ", "_").replace("(","").replace(")","")
        ph_result.summary.to_csv(
            f"{OUTPUT_DIR}/05_ph_check_final_{safe_label}.csv")
 
        # Schoenfeld residual plot for OS final model
        if ep_label == "Overall Survival (OS)":
            ax = cph.check_assumptions(subset, p_value_threshold=0.05,
                                       show_plots=True)
            plt.suptitle(f"Schoenfeld Residuals — {ep_label} Final Model",
                         fontsize=11, y=1.01)
            plt.tight_layout()
            plt.savefig(f"{OUTPUT_DIR}/05_schoenfeld_{safe_label}.png",
                        bbox_inches='tight')
            plt.close()
            print(f"  Schoenfeld plots → {OUTPUT_DIR}/05_schoenfeld_{safe_label}.png")
 
    except Exception as ex:
        print(f"  ⚠ PH test failed for {ep_label}: {ex}")

  Overall Survival (OS):
    Day.29.MRD                                    ✓ PH holds  p=0.0693

   Bootstrapping lowess lines. May take a moment...

Proportional hazard assumption looks okay.
  Schoenfeld plots → TARGET_multivariate_cox_outputs/05_schoenfeld_Overall_Survival_OS.png
  Event-Free Survival (EFS):
    Day.29.MRD                                    ⚠ VIOLATED  p=0.0000
  Disease-Free Survival (DFS):
    Day.29.MRD                                    ✓ PH holds  p=0.3998


#### STEP 6: MODEL PERFORMANCE

In [9]:
performance_rows = []
 
for ep_label, (cph, selected_vars, strata) in final_models.items():
    dur_col = next(d for d, e, l in ALL_ENDPOINTS if l == ep_label)
    evt_col = next(e for d, e, l in ALL_ENDPOINTS if l == ep_label)
 
    model_vars = selected_vars + (strata or [])
    subset = df_model[[c for c in model_vars + [dur_col, evt_col]
                        if c in df_model.columns]].dropna()
    for col in selected_vars:
        if col in subset.columns:
            subset[col] = pd.to_numeric(subset[col], errors='coerce')
    subset = subset.dropna()
 
    c_idx = cph.concordance_index_
    print(f"  {ep_label}: C-index = {c_idx:.3f}")
 
    performance_rows.append({
        "endpoint"     : ep_label,
        "n_patients"   : len(subset),
        "n_events"     : int(subset[evt_col].sum()),
        "n_vars_final" : len(selected_vars),
        "C_index"      : round(c_idx, 3),
        "selected_vars": ", ".join(selected_vars),
    })
 
    # ── Calibration at landmark time points ──
    median_t = subset[dur_col].median()
    t_landmarks = [t for t in [365, 3*365, 5*365] if t < subset[dur_col].max()]
    if not t_landmarks:
        t_landmarks = [median_t]
 
    fig, axes = plt.subplots(1, len(t_landmarks),
                              figsize=(len(t_landmarks) * 5, 4.5))
    if len(t_landmarks) == 1:
        axes = [axes]
 
    risk_scores = cph.predict_partial_hazard(subset)
    subset_plot = subset.copy()
    subset_plot['risk_score'] = risk_scores.values
 
    # Assign risk tertiles (low/mid/high) — more stable than deciles for n~200
    # WHY tertiles not deciles: each group needs enough events for KM estimate
    try: 
        subset_plot['risk_group'] = pd.qcut(
            subset_plot['risk_score'],
            q=3,
            labels=['Low', 'Mid', 'High'],
            duplicates='drop'   # silently merge duplicate edges
        )
    # If drop collapsed 3 bins → 2, the labels won't match — check
        n_groups = subset_plot['risk_group'].nunique()
        if n_groups < 3:
            raise ValueError(f"Only {n_groups} unique groups after dropping duplicates")
    except ValueError:
        # Fallback: median split into Low / High only
        print("  ⚠ Risk scores too concentrated for tertiles — falling back to median split")
        median_rs = subset_plot['risk_score'].median()
        subset_plot['risk_group'] = pd.cut(
            subset_plot['risk_score'],
            bins=[-np.inf, median_rs, np.inf],
            labels=['Low', 'High']
        )
 
    kmf_cal = KaplanMeierFitter()
    CAL_COLORS = {'Low': '#27AE60', 'Mid': '#F39C12', 'High': '#C0392B'}
 
    for ax, t_land in zip(axes, t_landmarks):
        for grp in ['Low', 'Mid', 'High']:
            mask = subset_plot['risk_group'] == grp
            if mask.sum() < 5:
                continue
            kmf_cal.fit(subset_plot.loc[mask, dur_col],
                        event_observed=subset_plot.loc[mask, evt_col],
                        label=f"{grp} risk (n={mask.sum()})")
            kmf_cal.plot_survival_function(ax=ax, ci_show=True,
                                           color=CAL_COLORS[grp],
                                           linewidth=2)
 
        ax.set_title(f"Predicted Risk Tertiles\n"
                     f"Landmark: {t_land//365:.0f} year(s)", fontsize=9)
        ax.set_xlabel("Time (days)", fontsize=8)
        ax.set_ylabel("Survival Probability", fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=8)
 
    safe_label = ep_label.replace(" ", "_").replace("(","").replace(")","")
    plt.suptitle(f"Risk Stratification — {ep_label} Final Model\n"
                 f"C-index={c_idx:.3f}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/06_calibration_{safe_label}.png",
                bbox_inches='tight')
    plt.close()
    print(f"  Calibration plot → {OUTPUT_DIR}/06_calibration_{safe_label}.png")
 
perf_df = pd.DataFrame(performance_rows)
perf_df.to_csv(f"{OUTPUT_DIR}/06_model_performance.csv", index=False)
print(f"\n  Performance table → {OUTPUT_DIR}/06_model_performance.csv")
print(perf_df[['endpoint', 'n_patients', 'n_events',
               'n_vars_final', 'C_index']].to_string(index=False))

  Overall Survival (OS): C-index = 0.637
  ⚠ Risk scores too concentrated for tertiles — falling back to median split
  Calibration plot → TARGET_multivariate_cox_outputs/06_calibration_Overall_Survival_OS.png
  Event-Free Survival (EFS): C-index = 0.664
  ⚠ Risk scores too concentrated for tertiles — falling back to median split
  Calibration plot → TARGET_multivariate_cox_outputs/06_calibration_Event-Free_Survival_EFS.png
  Disease-Free Survival (DFS): C-index = 0.587
  ⚠ Risk scores too concentrated for tertiles — falling back to median split
  Calibration plot → TARGET_multivariate_cox_outputs/06_calibration_Disease-Free_Survival_DFS.png

  Performance table → TARGET_multivariate_cox_outputs/06_model_performance.csv
                   endpoint  n_patients  n_events  n_vars_final  C_index
      Overall Survival (OS)        1328       141             1    0.637
  Event-Free Survival (EFS)        1328       220             1    0.664
Disease-Free Survival (DFS)        1013       141  

#### STEP 7: SUBGROUP ANALYSES (INTERACTION PLOTS)

We check here whether the key prognostic variables have consistent effects across clinically important subgroups (sex, ETP stsatus, age group). If the HR for MRD is very different in ETP vs. non-ETP patients, there may be a statistical interaction that the main model misses. 

This is presented as a "forest plot of forest plots" - each row is a subgroup, and the HR estimate in that subgroup is shown. Wide CIs in small subroups are expected - we look for qualitative consistencey (same direction), not exact magnitude.

In [10]:
# Define subgroups — use variables that were NOT in the final model
# (otherwise this becomes tautological)
SUBGROUP_VARS = {
    "Sex"       : {0: "Female", 1: "Male"},
    "ETP.STATUS": {0: "Non-ETP", 1: "ETP"},
}
# Age group: split at median
if "Age.at.Diagnosis.in.Years" in df_model.columns:
    age_median = df_model["Age.at.Diagnosis.in.Years"].median()
    df_model["Age_group"] = (
        df_model["Age.at.Diagnosis.in.Years"] >= age_median
    ).astype(int)
    SUBGROUP_VARS["Age_group"] = {
        0: f"Age < {age_median:.0f}y",
        1: f"Age ≥ {age_median:.0f}y"
    }
 
# Run subgroup analysis for primary endpoint (OS) final model
if "Overall Survival (OS)" in final_models:
    cph_final, final_vars_os, strata_os = final_models["Overall Survival (OS)"]
 
    # The PRIMARY variable of interest to check subgroup consistency
    # Pick the most significant variable from final OS model
    os_summary = final_summaries["Overall Survival (OS)"]
    if not os_summary.empty:
        primary_predictor_display = os_summary.sort_values('p_value').index[0]
        primary_predictor = display_to_col(
            primary_predictor_display.replace(".", " ").replace("_", " "),
            final_vars_os
        ) or primary_predictor_display
 
        print(f"  Primary predictor for subgroup analysis: {primary_predictor}")
 
        subgroup_rows = []
        valid_subgroups = {k: v for k, v in SUBGROUP_VARS.items()
                           if k in df_model.columns
                           and k != primary_predictor
                           and k not in (strata_os or [])}
 
        for sg_col, sg_labels in valid_subgroups.items():
            for sg_val, sg_name in sg_labels.items():
                mask = df_model[sg_col] == sg_val
                sg_df = df_model[mask].copy()
 
                if sg_df[PRIMARY_EVENT].sum() < 5:
                    print(f"  Skip {sg_name}: <5 events")
                    continue
 
                sg_vars = [primary_predictor]
                if primary_predictor not in sg_df.columns:
                    continue
 
                sg_subset = sg_df[[primary_predictor, PRIMARY_DURATION,
                                   PRIMARY_EVENT]].dropna()
                sg_subset[primary_predictor] = pd.to_numeric(
                    sg_subset[primary_predictor], errors='coerce'
                )
                sg_subset = sg_subset.dropna()
 
                if sg_subset[primary_predictor].nunique() < 2 or len(sg_subset) < 10:
                    continue
 
                try:
                    cph_sg = CoxPHFitter(penalizer=0.1)
                    cph_sg.fit(sg_subset,
                               duration_col=PRIMARY_DURATION,
                               event_col=PRIMARY_EVENT,
                               formula=safe_col_name(primary_predictor))
                    s = cph_sg.summary
                    hr  = float(s['exp(coef)'].values[0])
                    lo  = float(s['exp(coef) lower 95%'].values[0])
                    hi  = float(s['exp(coef) upper 95%'].values[0])
                    p   = float(s['p'].values[0])
 
                    subgroup_rows.append({
                        "subgroup"   : sg_name,
                        "n"          : len(sg_subset),
                        "n_events"   : int(sg_subset[PRIMARY_EVENT].sum()),
                        "HR"         : round(hr, 3),
                        "HR_lower95" : round(lo, 3),
                        "HR_upper95" : round(hi, 3),
                        "p_value"    : round(p, 4),
                    })
                    print(f"  {sg_name:<35} HR={hr:.3f} [{lo:.3f}–{hi:.3f}]  p={p:.4f}")
                except Exception as ex:
                    print(f"  {sg_name}: failed ({ex})")
 
        if subgroup_rows:
            sg_df_plot = pd.DataFrame(subgroup_rows)
            sg_df_plot.to_csv(f"{OUTPUT_DIR}/07_subgroup_analysis.csv", index=False)
 
            # Forest plot of subgroup HRs
            fig, (ax_tab, ax_for) = plt.subplots(
                1, 2, figsize=(13, max(5, len(sg_df_plot)*0.55 + 2.5)),
                gridspec_kw={'width_ratios': [2, 3]}
            )
            ax_tab.set_xlim(0, 1)
            ax_tab.set_ylim(-0.5, len(sg_df_plot) + 0.5)
            ax_tab.axis('off')
 
            ax_tab.text(0.0, len(sg_df_plot)+0.1, "Subgroup",
                        fontsize=9, fontweight='bold')
            ax_tab.text(0.60, len(sg_df_plot)+0.1, "n (events)",
                        fontsize=9, fontweight='bold')
            ax_tab.text(0.80, len(sg_df_plot)+0.1, "HR [95% CI]",
                        fontsize=9, fontweight='bold')
            ax_tab.axhline(len(sg_df_plot)-0.1, color='black', lw=0.8)
 
            ax_for.set_xscale('log')
            ax_for.set_ylim(-0.5, len(sg_df_plot)+0.5)
            ax_for.axvline(1.0, color='black', lw=1, ls='--')
 
            for i, row in sg_df_plot.iterrows():
                y = len(sg_df_plot) - 1 - i
                ax_tab.text(0.0, y, row['subgroup'], fontsize=8, va='center')
                ax_tab.text(0.60, y,
                            f"{row['n']} ({row['n_events']})",
                            fontsize=8, va='center', ha='center')
                ax_tab.text(0.80, y,
                            f"{row['HR']:.2f} [{row['HR_lower95']:.2f}–{row['HR_upper95']:.2f}]",
                            fontsize=8, va='center')
 
                ax_for.plot([max(row['HR_lower95'], 0.01),
                             min(row['HR_upper95'], 100)],
                            [y, y], color='#2C3E50', lw=1.8)
                ax_for.scatter(row['HR'], y, color='#2C3E50',
                               s=60, zorder=3, marker='s')
 
            ax_for.set_xlabel(f"HR for '{primary_predictor.replace('.', ' ')}'\n"
                              f"(log scale)", fontsize=9)
            ax_for.set_yticks([])
            ax_for.spines[['left', 'right', 'top']].set_visible(False)
 
            plt.suptitle(
                f"Subgroup Analysis — Effect of "
                f"'{primary_predictor.replace('.', ' ')}' on OS\n"
                f"Consistency check across clinical subgroups",
                fontsize=11, fontweight='bold', y=1.01
            )
            plt.tight_layout()
            plt.savefig(f"{OUTPUT_DIR}/07_subgroup_forest.png",
                        bbox_inches='tight')
            plt.close()
            print(f"\n  Subgroup forest plot → {OUTPUT_DIR}/07_subgroup_forest.png")

  Primary predictor for subgroup analysis: Day.29.MRD
  Female                              HR=1.024 [1.012–1.036]  p=0.0001
  Male                                HR=1.026 [1.016–1.036]  p=0.0000
  Skip Non-ETP: <5 events
  Skip ETP: <5 events
  Age < 9y                            HR=1.018 [1.006–1.031]  p=0.0046
  Age ≥ 9y                            HR=1.029 [1.020–1.039]  p=0.0000

  Subgroup forest plot → TARGET_multivariate_cox_outputs/07_subgroup_forest.png


#### STEP 8: EXPORT FINAL VARIABLE LIST FOR LASSO COX (R)

The multivariate Cox final model variables are the starting point for LASSO Cox in R. LASSO will further regularise within this set. 

In [11]:
# ── 8a. Final variable list per endpoint ──
for ep_label, (cph, selected_vars, strata) in final_models.items():
    safe_label = ep_label.replace(" ", "_").replace("(","").replace(")","")
    pd.DataFrame({
        "variable_col"   : selected_vars,
        "variable_display": [v.replace(".", " ").replace("_", " ")
                              for v in selected_vars],
        "strata"         : [(v in (strata or [])) for v in selected_vars],
    }).to_csv(f"{OUTPUT_DIR}/08_lasso_input_vars_{safe_label}.csv", index=False)
    print(f"  {ep_label} LASSO input vars → "
          f"{OUTPUT_DIR}/08_lasso_input_vars_{safe_label}.csv")
    print(f"    Variables: {selected_vars}")
 
# ── 8b. Numeric feature matrix (all final OS vars union) ──
all_final_vars = list(dict.fromkeys(
    v for (_, vars_list, _) in final_models.values()
    for v in vars_list
))
all_final_cols = [c for c in all_final_vars if c in df_model.columns]
 
# Impute and scale for glmnet
X_export = df_model[all_final_cols].copy()
for col in all_final_cols:
    X_export[col] = pd.to_numeric(X_export[col], errors='coerce')
 
imputer_exp = SimpleImputer(strategy='median')
X_exp_imp   = pd.DataFrame(
    imputer_exp.fit_transform(X_export),
    columns=all_final_cols, index=X_export.index
)
scaler_exp   = RobustScaler()
X_exp_scaled = pd.DataFrame(
    scaler_exp.fit_transform(X_exp_imp),
    columns=all_final_cols, index=X_exp_imp.index
)
 
# Add patient ID
if "USI" in df_model.columns:
    X_exp_scaled.insert(0, "USI", df_model["USI"].values)
 
X_exp_scaled.to_csv(f"{OUTPUT_DIR}/08_feature_matrix_for_lasso.csv", index=False)
print(f"\n  Feature matrix (scaled) → {OUTPUT_DIR}/08_feature_matrix_for_lasso.csv")
print(f"  Shape: {X_exp_scaled.shape}")
 
# ── 8c. Survival matrix ──
surv_cols = ["USI"] if "USI" in df_model.columns else []
for dur, evt, _ in ALL_ENDPOINTS:
    if dur in df_model.columns:
        surv_cols += [dur, evt]
 
surv_matrix = df_model[surv_cols].copy()
surv_matrix.to_csv(f"{OUTPUT_DIR}/08_survival_matrix_for_lasso.csv", index=False)
print(f"  Survival matrix → {OUTPUT_DIR}/08_survival_matrix_for_lasso.csv")
 
# ── R code snippet ──
r_vars = ", ".join(f'"{v}"' for v in all_final_cols)
r_snippet = f'''
# ============================================================
# R — LASSO Cox Regression (next step)
# ============================================================
library(glmnet)
library(survival)
 
# Load feature matrix and survival data
X_df   <- read.csv("TARGET_multivariate_cox_outputs/08_feature_matrix_for_lasso.csv")
surv_df <- read.csv("TARGET_multivariate_cox_outputs/08_survival_matrix_for_lasso.csv")
 
# Features for LASSO (final multivariate Cox variables)
feature_cols <- c({r_vars})
 
# Build glmnet inputs
X <- as.matrix(X_df[, feature_cols])   # predictor matrix — no NAs
y <- Surv(surv_df$OS, surv_df$OS.status)  # survival object
 
# Cross-validated LASSO Cox
set.seed(42)
cv_fit <- cv.glmnet(X, y, family = "cox", alpha = 1,
                    nfolds = 10, type.measure = "C")
 
# Optimal lambda
lambda_min <- cv_fit$lambda.min   # minimises CV error
lambda_1se <- cv_fit$lambda.1se   # 1-SE rule (more parsimonious)
 
# Extract non-zero coefficients at lambda.1se
coefs <- coef(cv_fit, s = "lambda.1se")
selected <- rownames(coefs)[coefs[,1] != 0]
cat("LASSO-selected variables:", selected, "\\n")
 
# Final LASSO Cox model coefficients
final_coefs <- as.data.frame(as.matrix(coefs))
final_coefs <- final_coefs[final_coefs[,1] != 0, , drop=FALSE]
print(final_coefs)
# ============================================================
'''
 
with open(f"{OUTPUT_DIR}/08_lasso_cox_starter.R", "w") as f:
    f.write(r_snippet)
print(f"\n  R starter script → {OUTPUT_DIR}/08_lasso_cox_starter.R")

  Overall Survival (OS) LASSO input vars → TARGET_multivariate_cox_outputs/08_lasso_input_vars_Overall_Survival_OS.csv
    Variables: ['Day.29.MRD']
  Event-Free Survival (EFS) LASSO input vars → TARGET_multivariate_cox_outputs/08_lasso_input_vars_Event-Free_Survival_EFS.csv
    Variables: ['Day.29.MRD']
  Disease-Free Survival (DFS) LASSO input vars → TARGET_multivariate_cox_outputs/08_lasso_input_vars_Disease-Free_Survival_DFS.csv
    Variables: ['Day.29.MRD']

  Feature matrix (scaled) → TARGET_multivariate_cox_outputs/08_feature_matrix_for_lasso.csv
  Shape: (1335, 2)
  Survival matrix → TARGET_multivariate_cox_outputs/08_survival_matrix_for_lasso.csv

  R starter script → TARGET_multivariate_cox_outputs/08_lasso_cox_starter.R


#### FINAL SUMMARY 

In [12]:
for ep_label in [l for _, _, l in ALL_ENDPOINTS]:
    if ep_label in final_models:
        _, vars_list, strata = final_models[ep_label]
        summ = final_summaries[ep_label]
        sig_count = summ['significant'].sum()
        c_idx = summ['C_index'].values[0] if len(summ) else "N/A"
        print(f"\n  {ep_label}:")
        print(f"    Final variables : {vars_list}")
        if strata:
            print(f"    Stratified on  : {strata}")
        print(f"    Significant (p<0.05): {sig_count}/{len(summ)}")
        print(f"    C-index        : {c_idx}")
 
print()
print("  Files written to:", OUTPUT_DIR)
outputs = [
    "01_vif_check.csv", "01_vif_chart.png",
    "02_full_model_*.csv",
    "03_aic_history_*.csv", "03_aic_trajectory_*.png",
    "04_forest_final_*.png", "04_final_model_*.csv",
    "05_ph_check_final_*.csv", "05_schoenfeld_*.png",
    "06_calibration_*.png", "06_model_performance.csv",
    "07_subgroup_analysis.csv", "07_subgroup_forest.png",
    "08_lasso_input_vars_*.csv",
    "08_feature_matrix_for_lasso.csv",
    "08_survival_matrix_for_lasso.csv",
    "08_lasso_cox_starter.R",
]
for f in outputs:
    print(f"  ├── {f}")
 
print()
print("  NEXT STEP → LASSO Cox Regression (R)")
print("  Use: TARGET_multivariate_cox_outputs/08_lasso_cox_starter.R")
print("       as the starting point for your R script")
print("=" * 70)
 


  Overall Survival (OS):
    Final variables : ['Day.29.MRD']
    Significant (p<0.05): 1/1
    C-index        : 0.637

  Event-Free Survival (EFS):
    Final variables : ['Day.29.MRD']
    Significant (p<0.05): 1/1
    C-index        : 0.664

  Disease-Free Survival (DFS):
    Final variables : ['Day.29.MRD']
    Significant (p<0.05): 1/1
    C-index        : 0.587

  Files written to: TARGET_multivariate_cox_outputs
  ├── 01_vif_check.csv
  ├── 01_vif_chart.png
  ├── 02_full_model_*.csv
  ├── 03_aic_history_*.csv
  ├── 03_aic_trajectory_*.png
  ├── 04_forest_final_*.png
  ├── 04_final_model_*.csv
  ├── 05_ph_check_final_*.csv
  ├── 05_schoenfeld_*.png
  ├── 06_calibration_*.png
  ├── 06_model_performance.csv
  ├── 07_subgroup_analysis.csv
  ├── 07_subgroup_forest.png
  ├── 08_lasso_input_vars_*.csv
  ├── 08_feature_matrix_for_lasso.csv
  ├── 08_survival_matrix_for_lasso.csv
  ├── 08_lasso_cox_starter.R

  NEXT STEP → LASSO Cox Regression (R)
  Use: TARGET_multivariate_cox_outputs/08